# RNA Seq Pipeline
#### Author: Kylie Gregory 
#### Created On: 11/25/2026
#### Last Modified: 11/25/2026
#### Purpose: Create an RNA-seq pipeline to determine what gene expression changes are consistent across two independent datasets comparing dilated cardiomyopathy (DCM) and healthy (non-failing donor) groups. 

#### Files: 


In [1]:
#load in libraries

library(DESeq2)
library(tidyverse)

Loading required package: S4Vectors



Loading required package: stats4



Loading required package: BiocGenerics



Loading required package: generics




Attaching package: ‘generics’




The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union




... (17 outputs left out)

The following object is masked from ‘package:MatrixGenerics’:

    rowMedians




The following objects are masked from ‘package:matrixStats’:

    anyMissing, rowMedians




── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ lubridate::%within%() masks IRanges::%within%()
✖ dplyr::collapse()     masks IRanges::collapse()
✖ dplyr::combine()      masks Biobase::combine(), BiocGenerics::combine()
✖ dplyr::count()        masks matrixStats::count()
✖ dplyr::desc()         masks IRanges::desc()
✖ tidyr::expand()       masks S4Vectors::expand()
✖ dplyr::filter()       masks stats::filter()
✖ dplyr::first()        masks S4Vectors::first()
✖ dplyr::lag()          masks stats::lag()
✖ ggplot2::Position()   masks BiocGenerics::Position(), base::Position()
✖ purrr::reduce()       masks GenomicRanges::reduce(), IRanges::reduce()
✖ dplyr::rename()       masks S4Vectors::rename()
✖ lubridate::second()   masks S4Vectors::second()
✖ lubridate::second<-() masks S4Vectors::second<-()
✖ dplyr::slice()        masks IRanges::slice()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
#read in RNA-seq raw counts dataset


create_raw_counts_table <- function(raw_counts_url, GEO_number){
    #either GSE141910.tsv.gz or GSE116250.tsv.gz
    file_name <- paste(GEO_number, ".tsv.gz", sep="")
    download.file(raw_counts_url, file_name)
    count_table <- read.table(file_name, header=T,sep="\t")
    #make GeneID column the unique row identifiers 
    count_table <- column_to_rownames(count_table, var ="GeneID")

    return(count_table)
    
}

In [3]:
d1_geo_number <- "GSE141910"


d1_count_url <- "https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc=GSE141910&format=file&file=GSE141910_raw_counts_GRCh38.p13_NCBI.tsv.gz"
GSE141910_count_table <- create_raw_counts_table(GSE141910_count_url, "GSE141910")

head(GSE141910_count_table)



,GSM4215858,GSM4215860,GSM4215861,GSM4215862,GSM4215863,GSM4215864,GSM4215865,GSM4215866,GSM4215867,GSM4215868,⋯,GSM4216214,GSM4216215,GSM4216216,GSM4216217,GSM4216218,GSM4216219,GSM4216221,GSM4216224,GSM4216226,GSM4216228
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
100287102,1,2,1,1,7,3,4,3,22,2,⋯,2,2,14,6,13,16,1,1,4,19
653635,249,463,218,181,517,285,199,315,1831,275,⋯,396,544,1041,1023,896,1064,192,181,376,1448
102466751,8,24,8,5,5,7,7,8,5,4,⋯,22,24,7,18,7,7,5,6,15,5
107985730,1,0,4,0,0,0,0,0,1,1,⋯,1,0,0,1,4,1,0,1,0,0
100302278,0,0,1,0,1,0,0,0,1,1,⋯,0,1,3,1,4,0,0,1,1,0
645520,1,1,1,0,2,0,1,0,4,2,⋯,2,4,1,2,6,4,0,3,8,1
